In [1]:
import os
import shutil
import sys
import json

import logging
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import numpy as np


from transformers import pipeline
import torch

In [2]:
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

In [3]:
# функция для преобразования сырого датасета в набор фотографий отдельного зуба
from utils_1_extract_teeth import extract_teeth

In [4]:
# простой CLIP-классификатор, нужен для работы extract_teeth
clip_classifier = pipeline(
    "zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=0 if torch.cuda.is_available() else -1
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-14 23:21:23,480 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


In [5]:
# проверяем, что работает создание датасета
#   - в test/debug/grid сохраняется задетектированная решетка линий на картоне
#   - в test/debug/accepted и test/debug/rejected сохраняются фото зубов и пустые соответсвенно

extract_teeth(
    input_folder="dataset_raw/11/11",
    output_folder="test",
    clip_classifier=clip_classifier,
    debug=True
)

  0%|          | 0/10 [00:00<?, ?it/s]

115

In [ ]:
# создаем чистый датасет, используя как одиночные, так и групповые фотографии

RAW_DATASET_DIR = 'dataset_raw'
CLEAN_DATASET_DIR = 'dataset_clean'

os.makedirs(CLEAN_DATASET_DIR, exist_ok=False)

tooth_id = 0 
teeth_classes = [f'{x}{y}' for y in range(1, 9) for x in range(1, 5)]
tooth_id_to_class = {}

for cls in teeth_classes:

    # single photos
    img_files = [
        f for f in os.listdir(f"{RAW_DATASET_DIR}/{cls}")
        if f.lower().endswith((".jpg",".jpeg",".png"))
    ]
    for file in img_files:
        shutil.copyfile(os.path.join(f"{RAW_DATASET_DIR}/{cls}", file), os.path.join(f"{CLEAN_DATASET_DIR}", f"{tooth_id:05d}.jpg"))
        tooth_id += 1
        tooth_id_to_class[tooth_id] = cls

    # group photos
    
    if os.path.exists(f"{RAW_DATASET_DIR}/{cls}/{cls}"):
        prev_tooth_id = tooth_id
        tooth_id = extract_teeth(
            input_folder=f"{RAW_DATASET_DIR}/{cls}/{cls}",
            output_folder=CLEAN_DATASET_DIR,
            clip_classifier=clip_classifier,
            debug=True,
            start_tooth_id=tooth_id,
        )
        for i in range(prev_tooth_id, tooth_id):
            tooth_id_to_class[i] = cls

with open(f"{CLEAN_DATASET_DIR}/tooth_id_to_class.json", "w") as f:
    json.dump(tooth_id_to_class, f)

print(f"total: {tooth_id} single tooth photos")

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

2026-03-14 23:22:20,568 [WARNING] 20260302_123438.jpg: incorrect line count H=4 V=3


  0%|          | 0/12 [00:00<?, ?it/s]

2026-03-14 23:22:25,584 [WARNING] 20260302_122406.jpg: incorrect line count H=4 V=3


  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/11 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

total: 910 single tooth photos


In [10]:
from collections import Counter

Counter(tooth_id_to_class.values())

Counter({'23': 199,
         '43': 168,
         '13': 127,
         '11': 115,
         '12': 102,
         '21': 94,
         '22': 93,
         '24': 2,
         '14': 1,
         '15': 1,
         '35': 1,
         '45': 1,
         '36': 1})